In [1]:
import pickle 
import pandas as pd

In [2]:
with open("cell/challenge_scores.pickle", "rb") as f:
    results = pickle.load(f)

df_results = pd.DataFrame(results).T
df_results["PaperProjectID"] = df_results.index
df_results = df_results[df_results["score"].isna()] # none means ok
df_results

,paper_ref,ref_ref,ref_spread,n_related,titles_only,status,score,PaperProjectID
0,0.096596,0.113608,0.043688,27,False,OK,NaN,0
1,0.095705,0.117094,0.046006,16,False,OK,NaN,1
2,0.09802,0.119422,0.039144,29,False,OK,NaN,2
3,0.094372,0.114645,0.037192,30,False,OK,NaN,3
4,0.094984,0.112259,0.041183,25,False,OK,NaN,4
...,...,...,...,...,...,...,...,...
6765,0.106573,0.12434,0.036197,53,False,OK,NaN,6765
6766,0.106351,0.128487,0.04007,26,False,OK,NaN,6766
6767,0.104461,0.119706,0.034577,64,False,OK,NaN,6767
6768,0.108723,0.140603,0.04451,45,False,OK,NaN,6768


In [3]:
df_cell = pd.read_csv("cell/cell.csv")[["OpenAlexID (as URL)", "PaperProjectID"]]
df_cell["openalex_id"] = df_cell["OpenAlexID (as URL)"].str.replace("https://openalex.org/", "")
df_cell = df_cell.drop(columns=["OpenAlexID (as URL)"])
df_cell

,PaperProjectID,openalex_id
0,0,W1965843055
1,1,W2046221312
2,2,W2150589615
3,3,W2087575288
4,4,W1989011389
...,...,...
6765,6765,W3111836713
6766,6766,W3104955345
6767,6767,W3105908160
6768,6768,W3093900830


In [4]:
df_results_with_id = df_results.merge(df_cell, on="PaperProjectID", how="left")
df_results_with_id

,paper_ref,ref_ref,ref_spread,n_related,titles_only,status,score,PaperProjectID,openalex_id
0,0.096596,0.113608,0.043688,27,False,OK,NaN,0,W1965843055
1,0.095705,0.117094,0.046006,16,False,OK,NaN,1,W2046221312
2,0.09802,0.119422,0.039144,29,False,OK,NaN,2,W2150589615
3,0.094372,0.114645,0.037192,30,False,OK,NaN,3,W2087575288
4,0.094984,0.112259,0.041183,25,False,OK,NaN,4,W1989011389
...,...,...,...,...,...,...,...,...,...
6571,0.106573,0.12434,0.036197,53,False,OK,NaN,6765,W3111836713
6572,0.106351,0.128487,0.04007,26,False,OK,NaN,6766,W3104955345
6573,0.104461,0.119706,0.034577,64,False,OK,NaN,6767,W3105908160
6574,0.108723,0.140603,0.04451,45,False,OK,NaN,6768,W3093900830


In [5]:
gold_scores = [
    "wangs_novelty",
    "dindex",
    "forward_citations_openalex",
    "novelty_sum",
    "new_finding",
    "originality"
] 
df_scores = pd.read_parquet("cell/cell_papers_all_novelty_scores.parquet")
df_scores = df_scores[["openalex"] + gold_scores]
df_scores.head()

,openalex,wangs_novelty,dindex,forward_citations_openalex,novelty_sum,new_finding,originality
0,W1965843055,NaN,-0.009080,711.0,1.0,1.0,0.855799
1,W2046221312,NaN,-0.004666,303.0,1.0,0.0,0.805782
2,W2150589615,NaN,-0.010714,820.0,8.0,7.0,0.890620
3,W2087575288,NaN,-0.007171,305.0,1.0,1.0,0.902148
4,W1989011389,NaN,-0.051230,1142.0,0.0,0.0,0.816867


In [6]:
df_all = df_results_with_id.merge(df_scores, left_on="openalex_id", right_on="openalex", how="left")
df_all.head()

,paper_ref,ref_ref,ref_spread,n_related,titles_only,status,score,PaperProjectID,openalex_id,openalex,wangs_novelty,dindex,forward_citations_openalex,novelty_sum,new_finding,originality
0,0.096596,0.113608,0.043688,27,False,OK,NaN,0,W1965843055,W1965843055,NaN,-0.009080,711.0,1.0,1.0,0.855799
1,0.095705,0.117094,0.046006,16,False,OK,NaN,1,W2046221312,W2046221312,NaN,-0.004666,303.0,1.0,0.0,0.805782
2,0.09802,0.119422,0.039144,29,False,OK,NaN,2,W2150589615,W2150589615,NaN,-0.010714,820.0,8.0,7.0,0.890620
3,0.094372,0.114645,0.037192,30,False,OK,NaN,3,W2087575288,W2087575288,NaN,-0.007171,305.0,1.0,1.0,0.902148
4,0.094984,0.112259,0.041183,25,False,OK,NaN,4,W1989011389,W1989011389,NaN,-0.051230,1142.0,0.0,0.0,0.816867


In [ ]:

def normalize(col):
    min_val = col.min()
    max_val = col.max()
    return (col - min_val) / (max_val - min_val)

def standardize(col):
    mean_val = col.mean()
    std_val = col.std()
    return (col - mean_val) / std_val


#df_all["score"] = (normalize(df_all["ref_ref"]) * normalize(df_all["paper_ref"]))/2
#df_all["score"] = (df_all["ref_ref"] * df_all["paper_ref"]).apply(
#    lambda x: x**(0.5) if x > 0 else 0
#)
#df_all["score"] = (
#   (normalize(df_all["ref_ref"]) * normalize(df_all["paper_ref"])).
#   apply(lambda x: x**(0.5) if x > 0 else 0)    
#   / (normalize(df_all["ref_ref"]) + normalize(df_all["paper_ref"]))
#)   
df_all["score_mean"] = (normalize(df_all["ref_ref"]) +  normalize(df_all["paper_ref"]))/2

df_all["score_product"] = (normalize(df_all["ref_ref"]) * normalize(df_all["paper_ref"])).apply(
    lambda x: x**(0.5) if x > 0 else 0
)
df_all["score_harmonic"] = 2 * (normalize(df_all["ref_ref"]) * normalize(df_all["paper_ref"])) / (normalize(df_all["ref_ref"]) + normalize(df_all["paper_ref"]))    

df_all["score_max"] = normalize(df_all["ref_ref"]).combine(normalize(df_all["paper_ref"]), max)

df_all["score_min"] = normalize(df_all["ref_ref"]).combine(normalize(df_all["paper_ref"]), min)


#df_all["score"] = normalize(df_all["ref_ref"]).combine(normalize(df_all["paper_ref"]), max)

def make_quantile(col):
    return col.apply(
        lambda x: (col <= x).mean()
    )

#df_all["score_quantile"] = make_quantile(df_all["ref_ref"])
#.combine(
#    make_quantile(df_all["paper_ref"]), max
#)

#df_all["score"] = (standardize(df_all["ref_ref"]).combine(standardize(df_all["paper_ref"]), max))

# from sklearn.linear_model import LinearRegression
# from sklearn.svm import SVR
# from sklearn.neural_network import MLPRegressor

# model = MLPRegressor(hidden_layer_sizes=(8, 4), max_iter=500, random_state=42, learning_rate="adaptive",
#                      activation="logistic", solver="adam", early_stopping=True, n_iter_no_change=10)
# inputs = ["ref_ref", "paper_ref", "ref_spread"]
# output = "new_finding"

# trainset = df_all[inputs + [output]].sample(frac=0.7)
# filter = trainset[inputs + [output]].notna().all(axis=1)

# X = trainset[inputs][filter]
# y = trainset[output][filter]

# model.fit(X, y)
# df_all["score_reg"] = model.predict(df_all[inputs]) 
# df_all["score_control"] = (
#     18.13580555 * df_all["ref_ref"] 
#     -11.33492044 * df_all["paper_ref"] 
#     -11.97745551 * df_all["ref_spread"]    
# )


#df_all["score"] = normalize(df_all["ref_ref"]) + normalize(df_all["paper_ref"])/2
#df_all["score"] = (normalize(df_all["ref_ref"]) + normalize(df_all["paper_ref"]) + normalize(df_all["ref_spread"]))/3
#df_all["score"] = normalize(df_all["ref_ref"]).combine(
#    normalize(df_all["paper_ref"]), max)
#.combine(normalize(df_all["ref_spread"]), max)


In [95]:
our_scores = [
    "paper_ref",
    "ref_ref",
    "ref_spread",
    "score",
    #"score_quantile",
    #score_reg",
    # "score_control"
]
df_all[our_scores + gold_scores].corr(method='spearman').loc[gold_scores][our_scores]

,paper_ref,ref_ref,ref_spread,score
wangs_novelty,0.283553,0.430183,0.130556,0.373733
dindex,0.307137,0.386447,0.147558,0.364745
forward_citations_openalex,-0.010173,0.034530,0.006167,0.013330
novelty_sum,0.041649,0.054712,0.005394,0.049724
new_finding,0.034797,0.034915,-0.001098,0.035201
originality,0.216567,0.355307,0.105817,0.301148


In [93]:
#model.coef_